# RCIS Scheduler
- Uses a FCFS (First Come, First Serve) Algorithm to determine interviewee bookings for companies

## Algo Approach
- For each day, get companies and interviewees available
- For each company in that day, get all timeslots where company is available.
- Filter interviewee list (form responses) for each timeslot + company preferred degree program, order based on response time
- Get interviewee at the top, remove interviewee from list


## Dependency Installation

## Imports

In [208]:
from datetime import datetime
from typing import NewType, Protocol
from enum import Enum
import pandas as pd

## Classes and Protocols for Scheduler

In [209]:
class DegreeProgram(str, Enum):
    ChemicalEngg = 'BS Chemical Engineering'
    CivilEngg = 'BS Civil Engineering'
    ComputerEngg = 'BS Computer Engineering'
    ComputerScience = 'BS Computer Science'
    ElectricalEngg = 'BS Electrical Engineering'
    ElectronicsEngg = 'BS Electronics Engineering'
    GeodeticEngg = 'BS Geodetic Engineering'
    IndustrialEngg = 'BS Industrial Engineering'
    MaterialsEngg = 'BS Materials Engineering'
    MechEngg = 'BS Mechanical Engineering'
    MetalEngg = 'BS Metallurgical Engineering'
    MiningEngg = 'BS Mining Engineering'

class Times(str, Enum): # not ideal, but I don't really want to deal with datetime right now
# Interview Simulations
    Time0900_0945 = "9:00-9:45 AM"
    Time1000_1045 = "10:00-10:45 AM"
    Time1100_1145 = "11:00-11:45 AM"
    Time1315_1400 = "1:15-2:00 PM"
    Time1415_1500 = "2:15-3:00 PM"
    Time1515_1600 = "3:15-4:00 PM"
    Time1615_1700 = "4:15-5:00 PM"

# Resume Consultations
    Time0900_0930 = "9:00-9:30 AM"
    Time0930_1000 = "9:30-10:00 AM"
    Time1015_1045 = "10:15-10:45 AM"
    Time1045_1115 = "10:45-11:15 AM"
    Time1130_1200 = "11:30 AM-12:00 PM"
    Time1330_1400 = "1:30-2:00 PM"
    Time1400_1430 = "2:00-2:30 PM"
    Time1445_1515 = "2:45-3:15 PM"
    Time1515_1545 = "3:15-3:45 PM"
    Time1600_1630 = "4:00-4:30 PM"
    Time1630_1700 = "4:30-5:00 PM"

coinciding_times_dict : dict[Times, list[Times]]= {
    Times.Time0900_0945: [Times.Time0900_0945, Times.Time0900_0930, Times.Time0930_1000],
    Times.Time1000_1045: [Times.Time1000_1045, Times.Time0930_1000, Times.Time1015_1045],
    Times.Time1100_1145: [Times.Time1100_1145, Times.Time1045_1115, Times.Time1130_1200],
    Times.Time1315_1400: [Times.Time1315_1400, Times.Time1330_1400],
    Times.Time1415_1500: [Times.Time1415_1500, Times.Time1400_1430, Times.Time1445_1515],
    Times.Time1515_1600: [Times.Time1515_1600, Times.Time1445_1515, Times.Time1515_1545, Times.Time1600_1630],
    Times.Time1615_1700: [Times.Time1615_1700, Times.Time1600_1630, Times.Time1630_1700],
    Times.Time0900_0930: [Times.Time0900_0930, Times.Time0900_0945],
    Times.Time0930_1000: [Times.Time0930_1000, Times.Time0900_0945, Times.Time1000_1045],
    Times.Time1015_1045: [Times.Time1015_1045, Times.Time1000_1045],
    Times.Time1045_1115: [Times.Time1045_1115, Times.Time1100_1145],
    Times.Time1130_1200: [Times.Time1130_1200, Times.Time1100_1145],
    Times.Time1330_1400: [Times.Time1330_1400, Times.Time1315_1400],
    Times.Time1400_1430: [Times.Time1400_1430, Times.Time1415_1500],
    Times.Time1445_1515: [Times.Time1445_1515, Times.Time1415_1500, Times.Time1515_1600],
    Times.Time1515_1545: [Times.Time1515_1545, Times.Time1515_1600],
    Times.Time1600_1630: [Times.Time1600_1630, Times.Time1515_1600, Times.Time1615_1700],
    Times.Time1630_1700: [Times.Time1630_1700, Times.Time1615_1700]
}
class Participant(Protocol):
    @property
    def name(self) -> str:
        # Return str that is the name of the participant (could be interviewee/interviewer)
        ...
    @property
    def time_slots(self) -> list[Times]:
        # Returns time slots of participant
        ...
class Interviewee:
    def __init__(self, name: str, email: str, capes_id: str):
        self._name = name
        self._email = email
        self._capes_id = capes_id
        self._booked_times: dict[str, dict[str,Times]] = {"November 13" : {"RC" : None, "IS" : None}, "November 14" : {"RC" : None, "IS" : None}}
    def __eq__(self, b):
        return type(self) == type(b) and self.name == b.name 
    def __hash__(self):
        return hash(self.name)
    def __str__(self):
        return f"Interviewee: {self._name}"
    @property
    def name(self) -> str:
        return self._name
    @property
    def email(self) -> str:
        return self._email
    def get_booked_times(self, day : str, type : str) -> Times:
        return self._booked_times[day][type]
    @property
    def capes_id(self) -> str:
        return self._capes_id
    def add_time(self, time : Times, type : str, day : str):
        self._booked_times[day][type] = time
    def check_coinciding_times(self, time : Times, type : str, day : str):
        try:
            return 0 if time in coinciding_times_dict[self._booked_times[day]['RC' if type == 'IS' else 'IS']] else 1
        except KeyError:
            return 1 

class Interviewer:
    def __init__(self, name: str, time_slots: list[Times], degree_program_preference: list[DegreeProgram] | DegreeProgram, appointment_type : str, interviewer_capacity_per_timeslot : int = 2):
        self._name = name
        self._time_slots = time_slots
        self._degree_program_preference = degree_program_preference
        self._interviewee_list : dict[Times, list[Interviewee]] = dict()
        self._appointment_type = appointment_type
        self._interviewee_capacity_per_timeslot = interviewer_capacity_per_timeslot
    def __lt__(a : Interviewer, b : Interviewer):
        return a.priority < b.priority
    def unallocated_type(self, type):
        self._unallocated_type = type
    @property
    def name(self) -> str:
        return self._name
    @property
    def time_slots(self) -> list[Times]:
        return self._time_slots
    @property
    def degree_program_preference(self) -> list[DegreeProgram] | DegreeProgram:
        return self._degree_program_preference
    @property
    def interviewee_list(self) -> dict[Times, list[Interviewee]]:
        return self._interviewee_list
    @property
    def priority(self):
        return len(self._time_slots)
    @property
    def appointment_type(self):
        return self._appointment_type
    @property
    def interviewee_capacity_per_timeslot(self):
        return self._interviewee_capacity_per_timeslot
    def add_to_interviewee_list(self, interviewee: Interviewee, time : Times):
        if time not in self._time_slots:
            return 0
        if time not in self._interviewee_list.keys():
            self._interviewee_list[time] = []
        # print(self._interviewee_list)
        if len(self._interviewee_list[time]) >= self._interviewee_capacity_per_timeslot:
            return 0
        self._interviewee_list[time].append(interviewee)
        return 1    
    

## Get Interviewe/Response Data

In [210]:
responses_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRfqv-xOqEipSS3Y-tY899C4yCAJWHrpucXflS6K8DPueZRNVspVh5BgDE24khojeqj1hAKtZ9GFj1F/pub?gid=1766557397&single=true&output=csv"
capes_card_url_2526 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQa39d0BnFMf0GB_NdAHbON6clsrSVolPytpl5JJQRFG02kI_l5cZZbRhQfwvFj0L-ukZRNxi5AEW7A/pub?gid=0&single=true&output=csv"
capes_card_url_2425 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQa39d0BnFMf0GB_NdAHbON6clsrSVolPytpl5JJQRFG02kI_l5cZZbRhQfwvFj0L-ukZRNxi5AEW7A/pub?gid=978666763&single=true&output=csv"

interviewee_df = pd.read_csv(responses_url)
capes_card_df = pd.read_csv(capes_card_url_2526)
capes_card_2425_df = pd.read_csv(capes_card_url_2425)

capes_card_df = pd.concat([capes_card_df, capes_card_2425_df[['CAPES CARD', 'LAST NAME', 'FIRST NAME', 'MI', 'COURSE','YR STANDING']]]).drop_duplicates()

In [211]:
capes_card_df.head()

,CAPES CARD,LAST NAME,FIRST NAME,MI,COURSE,YR STANDING
0,jachua,Chua,John Victor,A.,BS Industrial Engineering,4th
1,tlarcalas,Arcalas,Terence,L.,BS HE,3rd
2,tsngo,Ngo,Trei Sebastian,S.,BS Industrial Engineering,3rd
3,amtolentino,Tolentino,Andrea Nichole,M.,BS Computer Engineering,3rd
4,aamatsuzaki,Matsuzaki,Aryssa,A.,BS Computer Engineering,3rd


## Data Cleaning

### Column Selection for Cleaning

In [212]:
# RC/IS date column name
date_column_filter = 'What is your most preferred date?'



In [213]:
# Get columns to clean

# Remove duplicate columns
interviewee_df = interviewee_df[list(filter(lambda x : ".1" not in x, interviewee_df.columns))]

int_cols = interviewee_df.select_dtypes(include='float').columns
rcis_date_cols = interviewee_df.columns[interviewee_df.columns.str.contains(date_column_filter)]
# Remove NaN values
interviewee_df[int_cols] = interviewee_df[int_cols].fillna(12).astype('int') 
interviewee_df[rcis_date_cols] = interviewee_df[rcis_date_cols].fillna('None')
interviewee_df['Degree Program'] = interviewee_df['CAPES ID'].map(capes_card_df.set_index('CAPES CARD')['COURSE'])
interviewee_df['Name'] = interviewee_df['CAPES ID'].map(capes_card_df.set_index('CAPES CARD')['FIRST NAME']) + ' ' + interviewee_df['CAPES ID'].map(capes_card_df.set_index('CAPES CARD')['LAST NAME']) 

# sorry tinamad
old_string = 'What are your preferred time slots for RC, November 13 (Thursday)? '
new_string = 'RC Slot November 13 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

old_string = 'What are your preferred time slots for IS, November 13 (Thursday)? '
new_string = 'IS Slot November 13 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

old_string = 'What are your preferred time slots for RC, November 14 (Friday)? '
new_string = 'RC Slot November 14 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

old_string = 'What are your preferred time slots for IS, November 14 (Friday)? '
new_string = 'IS Slot November 14 - '
interviewee_df.columns = interviewee_df.columns.str.replace(old_string, new_string, regex=False)

interviewee_df.head()

,Timestamp,Data Privacy Agreement,CAPES ID,Are you representing any UPD College of Engineering Organization in joining this event?,Updated Resume or CV,What sub-event would you like to register for?,What is your most preferred date? [Resume Consultation (RC)],What is your most preferred date? [Interview Simulation (IS)],RC Slot November 13 - [9:00-9:30 AM],RC Slot November 13 - [9:30-10:00 AM],...,IS Slot November 14 - [3:15-4:00 PM],IS Slot November 14 - [4:15-5:00 PM],Email Address,Unnamed: 81,Email Addresses,Unnamed: 83,Unnamed: 84,Unnamed: 85,Degree Program,Name
0,11/2/2025 20:03:04,I agree,xmjaudalso,No,https://drive.google.com/open?id=1MqWge49TDvC0...,Resume Consultations (RC) ONLY,November 13 (Thu),None,12,12,...,12,12,NaN,12,xmjaudalso@up.edu.ph,12,BS Civil Engineering,Graduate,BS Civil Engineering,Xander Aia Danina Jaudalso
1,11/4/2025 16:38:15,I agree,cecarpio,No,https://drive.google.com/open?id=1J7RqQFHnOZMs...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,2,3,NaN,12,cecarpio@up.edu.ph,12,BS Electronics Engineering,4th,BS Electronics Engineering,Cyress Lein Carpio
2,11/6/2025 0:57:43,I agree,mdpunzalan,No,https://drive.google.com/open?id=1gkdZswnlbqPg...,Both (RC & IS),November 13 (Thu),November 13 (Thu),12,12,...,12,12,NaN,12,marianne.punzalan@eee.upd.edu.ph,12,BS Computer Engineering,4th,BS Computer Engineering,Marianne Veronica Punzalan
3,11/6/2025 12:33:38,I agree,mmurillo,No,https://drive.google.com/open?id=18hwpPcI2noW1...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,4,5,NaN,12,mmurillo1@up.edu.ph,12,BS Materials Engineering,3rd,BS Materials Engineering,Maria Romela Murillo
4,11/6/2025 14:42:04,I agree,cmborrega,No,https://drive.google.com/open?id=1eND6TnRl__uo...,Both (RC & IS),November 13 (Thu),November 13 (Thu),12,12,...,12,12,NaN,12,cmborrega@up.edu.ph,12,BS Computer Engineering,4th,BS Computer Engineering,Clyde Lawrence Borrega


## Scheduler

- Sort interviewees by increasing rank and timestamp for a certain day and timeslot
- Get a company
- Filter out interviewee list based on company requirements (course, etc.)
- FCFS picking
- Remove chosen interviewees from list

In [214]:
rc_date_filter = ['RC, November 13', 'RC, November 14']
is_date_filter = ['IS, November 13', 'IS, November 14']

columns = interviewee_df.columns[interviewee_df.columns.str.contains(is_date_filter[0])]

interviewee_df = interviewee_df.sort_values(by=['Timestamp'], ascending=True)

interviewee_df[['Degree Program', 'Name']] = interviewee_df[['Degree Program', 'Name']].fillna('Not in Database')

interviewee_df.head()

,Timestamp,Data Privacy Agreement,CAPES ID,Are you representing any UPD College of Engineering Organization in joining this event?,Updated Resume or CV,What sub-event would you like to register for?,What is your most preferred date? [Resume Consultation (RC)],What is your most preferred date? [Interview Simulation (IS)],RC Slot November 13 - [9:00-9:30 AM],RC Slot November 13 - [9:30-10:00 AM],...,IS Slot November 14 - [3:15-4:00 PM],IS Slot November 14 - [4:15-5:00 PM],Email Address,Unnamed: 81,Email Addresses,Unnamed: 83,Unnamed: 84,Unnamed: 85,Degree Program,Name
29,11/11/2025 14:38:42,I agree,dfvalenzuela,UP IE Club,https://drive.google.com/open?id=15B5K8MxXusaU...,Both (RC & IS),November 13 (Thu),November 13 (Thu),12,12,...,12,12,dfvalenzuela@up.edu.ph,12,dfvalenzuela@up.edu.ph,12,NaN,NaN,BS Industrial Engineering,David Alfonso Valenzuela
30,11/11/2025 14:58:38,I agree,egalfaro,No,https://drive.google.com/open?id=1emeBVbzOqdCm...,Resume Consultations (RC) ONLY,November 14 (Fri),None,12,12,...,12,12,egalfaro@up.edu.ph,12,egalfaro@up.edu.ph,12,NaN,NaN,BS Industrial Engineering,Erinne Alfaro
31,11/11/2025 16:45:17,I agree,aearenas,UP ERG,https://drive.google.com/open?id=1QOy_dsyOqsmc...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,2,3,aearenas@up.edu.ph,12,aearenas@up.edu.ph,12,NaN,NaN,BS Computer Engineering,Arvin Jayson Arenas
32,11/11/2025 17:06:23,I agree,dbdee,No,https://drive.google.com/open?id=1T3Cb4vbrNQa1...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,2,3,dbdee@up.edu.ph,12,dbdee@up.edu.ph,12,NaN,NaN,BS Computer Science,Denise Mae Dee
33,11/11/2025 18:18:57,I agree,assantiago,UP ERG,https://drive.google.com/open?id=1cci_pnVjVmjP...,Both (RC & IS),November 13 (Thu),November 14 (Fri),12,12,...,1,3,assantiago3@up.edu.ph,12,assantiago3@up.edu.ph,12,NaN,NaN,BS Electronics Engineering,Angelo Gabriel Santiago


### Create Interviewer Dictionary for to Denote Availability

In [215]:
interviewer_availability_dict : dict[str, dict[str, list[Interviewer]]]= {
    'November 13' : {
        'RC' : [
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='RC'
            ),
            Interviewer(
                name='GHD',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                    Times.Time1330_1400,
                    Times.Time1400_1430
                ],
                degree_program_preference=[
                    DegreeProgram.CivilEngg,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.GeodeticEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MiningEngg
                ],
                appointment_type='RC'
            )
        ],
        'IS' : [
            Interviewer(
                name='GHD',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                    Times.Time1315_1400, 
                    Times.Time1415_1500
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='IS',
                interviewer_capacity_per_timeslot=1
            ),
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='IS'
            )
        ]
    },
    'November 14' : {
        'RC' : [
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='RC'
            ),
            Interviewer(
                name='Concepcion Industrial Corporation',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                    Times.Time1330_1400,
                    Times.Time1400_1430,
                    Times.Time1445_1515,
                    Times.Time1515_1545,
                    Times.Time1600_1630,
                    Times.Time1630_1700
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MaterialsEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='RC'
            ),
            Interviewer(
                name='Seven Seven Global Services Inc.',
                time_slots=[
                    Times.Time0900_0930, 
                    Times.Time0930_1000, 
                    Times.Time1015_1045, 
                    Times.Time1045_1115, 
                    Times.Time1130_1200,
                    Times.Time1330_1400,
                    Times.Time1400_1430,
                    Times.Time1445_1515,
                    Times.Time1515_1545,
                    Times.Time1600_1630,
                    Times.Time1630_1700
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='RC'
            )
        ],
        'IS' : [
            Interviewer(
                name='Seven Seven Global Services Inc.',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                    Times.Time1315_1400, 
                    Times.Time1415_1500,
                    Times.Time1515_1600,
                    Times.Time1615_1700,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='IS',
                interviewer_capacity_per_timeslot=1
            ),
            Interviewer(
                name='Huawei Technologies Phils. Inc.',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                ],
                appointment_type='IS',
                interviewer_capacity_per_timeslot=2
            ),
            Interviewer(
                name='Concepcion Industrial Coproration',
                time_slots=[
                    Times.Time0900_0945, 
                    Times.Time1000_1045, 
                    Times.Time1100_1145, 
                    Times.Time1315_1400, 
                    Times.Time1415_1500,
                    Times.Time1515_1600,
                    Times.Time1615_1700,
                ],
                degree_program_preference=[
                    DegreeProgram.ComputerEngg,
                    DegreeProgram.ComputerScience,
                    DegreeProgram.ElectricalEngg,
                    DegreeProgram.ElectronicsEngg,
                    DegreeProgram.IndustrialEngg,
                    DegreeProgram.MechEngg
                ],
                appointment_type='IS',
                interviewer_capacity_per_timeslot=1
            )
        ]
    }
}

In [216]:
interviewee_df.sort_values(by='RC Slot November 13 - [9:00-9:30 AM]')

interviewee_df[interviewee_df['CAPES ID'] == 'mrsales'][interviewee_df.columns[interviewee_df.columns.str.contains('IS Slot November 13')]]

,IS Slot November 13 - [9:00-9:45 AM],IS Slot November 13 - [10:00-10:45 AM],IS Slot November 13 - [11:00-11:45 AM],IS Slot November 13 - [1:15-2:00 PM],IS Slot November 13 - [2:15-3:00 PM],IS Slot November 13 - [3:15-4:00 PM],IS Slot November 13 - [4:15-5:00 PM]
37,12,12,12,12,3,2,1


## Algo Function

In [217]:
def get_booked_interviewee(name : str, interviewees : list[Interviewee]):
    for x in interviewees:
        if x.name == name:
            return x

def scheduling_algorithm(day: str, type: str, filtered_df: pd.DataFrame, booked_interviewees : list[Interviewee], override_skip : bool):
    booked_interviewees_loc = booked_interviewees
    for interviewer in interviewer_availability_dict[day][type]:
        print(f"Interviewer: {interviewer.name}; Target Program: {interviewer.degree_program_preference}")
        filter_by_desired_course = filtered_df[filtered_df['Degree Program'].isin(interviewer.degree_program_preference)]
        print(f"Interviewees: {len(filter_by_desired_course)}")
        for idx, row in filter_by_desired_course.iterrows():
            print(f"Interviewee Info: \n Name: {row['Name']}\n Course: {row['Degree Program']}")
            # Create interviewee instance if not existing
            if row['Name'] not in [interviewee.name for interviewee in booked_interviewees_loc]:
                interviewee_instance = Interviewee(name=row['Name'] , capes_id=row['CAPES ID'], email="") 
            else:
                for i in booked_interviewees_loc:
                    if i.name == row['Name']:
                        print(f"Found previous instance of {i.name}. Using that instead")
                        interviewee_instance = i
                        break
                if interviewee_instance.get_booked_times(day, type) != None:
                    print(f"Skipped {row['Name']} because they already have a booking.\n")
                    continue
            # get scores of interviewee/response
            scores_series = row[filter_by_desired_course.columns[filter_by_desired_course.columns.str.contains(f'{type} Slot {day}')]]
            melted_df = scores_series.to_frame().T.melt(
                ignore_index=False,
                var_name='Time Slot',
                value_name='Ranking'
            ).reset_index(drop=True).sort_values(by='Ranking', ascending=True)

            # clean scores to map to Timeslot enum values
            melted_df['Time Slot'] = melted_df['Time Slot'].str.replace(f'{type} Slot {day} - ', '', regex=False)
            melted_df_filtered = melted_df[melted_df['Ranking'] != 12].copy()
            melted_df_filtered['Time Slot Clean'] = melted_df_filtered['Time Slot'].str.strip('[]')
            # Create the reverse mapping dictionary: Time String -> Enum Member
            enum_map = {member.value: member for member in Times}
            melted_df_filtered['Time Enum Member'] = melted_df_filtered['Time Slot Clean'].map(enum_map)
            # map to interviewer with available timeslot
            for timeslot in melted_df_filtered['Time Enum Member']: 
                if(interviewee_instance.check_coinciding_times(timeslot, type, day) == 1 and interviewer.add_to_interviewee_list(interviewee_instance, timeslot) == 1):
                    print(f"Booked {interviewee_instance.name} to {timeslot}\n")
                    interviewee_instance.add_time(timeslot, type=type, day=day)
                    filter_by_desired_course.drop(idx)
                    if interviewee_instance.name not in [i.name for i in booked_interviewees_loc]:
                        booked_interviewees_loc.append(interviewee_instance)   
                    break
                else:
                    print(f"Not appointed for {timeslot}")
        print("----------------------------------------\n\n")
    filtered_df = filtered_df[~filtered_df['Name'].isin([interviewee.name for interviewee in booked_interviewees_loc])]
    return filtered_df, booked_interviewees_loc

def log_scheduler(day : str, type : str):
    for interviewer in interviewer_availability_dict[day][type]:
        print(f"----Interview Schedule for {interviewer.name}----")

        for timeslot in interviewer.time_slots:
            print(f"Timeslot: {timeslot}")
            try: 
                for interviewee in interviewer.interviewee_list[timeslot]:
                    print(interviewee)
            except KeyError:
                print("No interviewees for the timeslot.")
        print("---------------------")

## November 13 RC

In [218]:
# filter_first_day_rc = interviewee_df[interviewee_df['What is your most preferred date?  [Resume Consultation (RC)]'].str.contains('November 13')]
filter_first_day_rc = interviewee_df.sort_values(by='What is your most preferred date?  [Resume Consultation (RC)]', key= lambda col: ~col.str.contains('November 13', na=False))
carryover_interviewees_rc, booked_interviewees_rc = scheduling_algorithm(day='November 13', type='RC', filtered_df=filter_first_day_rc, booked_interviewees=[], override_skip=0)

Interviewer: Huawei Technologies Phils. Inc.; Target Program: [<DegreeProgram.ComputerEngg: 'BS Computer Engineering'>, <DegreeProgram.ComputerScience: 'BS Computer Science'>, <DegreeProgram.ElectricalEngg: 'BS Electrical Engineering'>, <DegreeProgram.ElectronicsEngg: 'BS Electronics Engineering'>]
Interviewees: 18
Interviewee Info: 
 Name: John Louis Nieto
 Course: BS Electronics Engineering
Booked John Louis Nieto to Times.Time0930_1000

Interviewee Info: 
 Name: Denzell Robyn Dy
 Course: BS Computer Science
Booked Denzell Robyn Dy to Times.Time1130_1200

Interviewee Info: 
 Name: Marianne Veronica Punzalan
 Course: BS Computer Engineering
Booked Marianne Veronica Punzalan to Times.Time1045_1115

Interviewee Info: 
 Name: Clyde Lawrence Borrega
 Course: BS Computer Engineering
Booked Clyde Lawrence Borrega to Times.Time1015_1045

Interviewee Info: 
 Name: Stephen James Gonda
 Course: BS Computer Science
Booked Stephen James Gonda to Times.Time1130_1200

Interviewee Info: 
 Name: Czar

In [219]:
log_scheduler('November 13', 'RC')

----Interview Schedule for Huawei Technologies Phils. Inc.----
Timeslot: Times.Time0900_0930
Interviewee: Ehren Castillo
Interviewee: Dominic Ian Caringal
Timeslot: Times.Time0930_1000
Interviewee: John Louis Nieto
Interviewee: Shaira Mae Rodriguez
Timeslot: Times.Time1015_1045
Interviewee: Clyde Lawrence Borrega
Interviewee: Francis Martin Bernales
Timeslot: Times.Time1045_1115
Interviewee: Marianne Veronica Punzalan
Interviewee: Czar Timothy Vinson Aguilar
Timeslot: Times.Time1130_1200
Interviewee: Denzell Robyn Dy
Interviewee: Stephen James Gonda
---------------------
----Interview Schedule for GHD----
Timeslot: Times.Time0900_0930
Interviewee: Jerd Carlos
Interviewee: Renato Jr. Tongko
Timeslot: Times.Time0930_1000
Interviewee: Sherwin Noynay
Timeslot: Times.Time1015_1045
Interviewee: Shawn Lim Gulayan
Interviewee: Jervi Elijah dela Cruz
Timeslot: Times.Time1045_1115
Interviewee: Juan Miguel Azul
Interviewee: Alyanna Ysabel Nuqui
Timeslot: Times.Time1130_1200
Interviewee: David Alf

## November 13 IS


In [220]:
filter_first_day_is = interviewee_df.sort_values(by='What is your most preferred date?  [Interview Simulation (IS)]', key=lambda col: ~col.str.contains('November 13', na=False))

carryover_interviewees_is, booked_interviewees_is = scheduling_algorithm('November 13', 'IS', filter_first_day_is, booked_interviewees=booked_interviewees_rc, override_skip=True)

log_scheduler('November 13', 'IS')
print(booked_interviewees_is)


Interviewer: GHD; Target Program: [<DegreeProgram.ComputerEngg: 'BS Computer Engineering'>, <DegreeProgram.ComputerScience: 'BS Computer Science'>, <DegreeProgram.ElectricalEngg: 'BS Electrical Engineering'>, <DegreeProgram.ElectronicsEngg: 'BS Electronics Engineering'>, <DegreeProgram.IndustrialEngg: 'BS Industrial Engineering'>, <DegreeProgram.MechEngg: 'BS Mechanical Engineering'>]
Interviewees: 35
Interviewee Info: 
 Name: David Alfonso Valenzuela
 Course: BS Industrial Engineering
Found previous instance of David Alfonso Valenzuela. Using that instead
Booked David Alfonso Valenzuela to Times.Time1315_1400

Interviewee Info: 
 Name: Marianne Veronica Punzalan
 Course: BS Computer Engineering
Found previous instance of Marianne Veronica Punzalan. Using that instead
Not appointed for Times.Time1515_1600
Not appointed for Times.Time1615_1700
Interviewee Info: 
 Name: Denzell Robyn Dy
 Course: BS Computer Science
Found previous instance of Denzell Robyn Dy. Using that instead
Not appoi

## November 14 RC

In [221]:
filter_second_day_rc = interviewee_df.sort_values(by='What is your most preferred date?  [Resume Consultation (RC)]', key= lambda col: ~col.str.contains('November 14', na=False))
print([interviewee.name for interviewee in booked_interviewees_rc])
print(len(filter_second_day_rc))

['John Louis Nieto', 'Denzell Robyn Dy', 'Marianne Veronica Punzalan', 'Clyde Lawrence Borrega', 'Stephen James Gonda', 'Czar Timothy Vinson Aguilar', 'Francis Martin Bernales', 'Shaira Mae Rodriguez', 'Ehren Castillo', 'Dominic Ian Caringal', 'David Alfonso Valenzuela', 'Juan Miguel Azul', 'Jerd Carlos', 'Leevan Hernandez ', 'Khymbee Gaa', 'Sherwin Noynay', 'Alyanna Ysabel Nuqui', 'Xander Aia Danina Jaudalso', 'Shawn Lim Gulayan', 'Erinne Alfaro', 'Jervi Elijah dela Cruz', 'John Erickson Lao', 'Renato Jr. Tongko']
53


In [222]:
filter_second_day_rc_new = filter_second_day_rc[
    ~filter_second_day_rc['Name'].isin([interviewee.name for interviewee in booked_interviewees_rc])
]
filter_second_day_rc_new.groupby(by='Degree Program')['Degree Program'].value_counts()

Degree Program
BS Business Administration      1
BS Chemical Engineering         5
BS Chemistry                    2
BS Civil Engineering            2
BS Computer Engineering         2
BS Computer Science             2
BS Electronics Engineering      4
BS Industrial Engineering       4
BS Materials Engineering        2
BS Mechanical Engineering       1
BS Metallurgical Engineering    2
Not in Database                 3
Name: count, dtype: int64

In [223]:
filter_second_day_rc_new = filter_second_day_rc_new.sort_values(by=['Degree Program'], key=lambda col: ~(col.str.contains('Industrial', na=False))) # exception made to prio IE to balance it out
carryover_interviewees, booked_interviewees_rc = scheduling_algorithm('November 14', 'RC', filter_second_day_rc_new, booked_interviewees_rc, override_skip=False)

log_scheduler('November 14', 'RC')


Interviewer: Huawei Technologies Phils. Inc.; Target Program: [<DegreeProgram.ComputerEngg: 'BS Computer Engineering'>, <DegreeProgram.ComputerScience: 'BS Computer Science'>, <DegreeProgram.ElectricalEngg: 'BS Electrical Engineering'>, <DegreeProgram.ElectronicsEngg: 'BS Electronics Engineering'>]
Interviewees: 8
Interviewee Info: 
 Name: Angelo Gabriel Santiago
 Course: BS Electronics Engineering
Not appointed for Times.Time1445_1515
Not appointed for Times.Time1515_1545
Interviewee Info: 
 Name: Denise Mae Dee
 Course: BS Computer Science
Not appointed for Times.Time1445_1515
Not appointed for Times.Time1515_1545
Not appointed for Times.Time1600_1630
Not appointed for Times.Time1630_1700
Not appointed for Times.Time1400_1430
Not appointed for Times.Time1330_1400
Interviewee Info: 
 Name: Khael Nikolas Perez
 Course: BS Computer Engineering
Not appointed for Times.Time1330_1400
Interviewee Info: 
 Name: Arvin Jayson Arenas
 Course: BS Computer Engineering
Not appointed for Times.Time

In [224]:
carryover_interviewees.to_csv('free_rc.csv')


## November 14 IS

In [225]:
filter_second_day_is = interviewee_df.sort_values(by='What is your most preferred date?  [Interview Simulation (IS)]', key=lambda col: ~col.str.contains('November 14', na=False))


carryover_interviewees_is, booked_interviewees_is = scheduling_algorithm('November 14', 'IS', filter_first_day_is, booked_interviewees_is, override_skip=False)

log_scheduler('November 14', 'IS')

carryover_interviewees_is.to_csv('free_is.csv')


Interviewer: Seven Seven Global Services Inc.; Target Program: [<DegreeProgram.ComputerEngg: 'BS Computer Engineering'>, <DegreeProgram.ComputerScience: 'BS Computer Science'>, <DegreeProgram.ElectricalEngg: 'BS Electrical Engineering'>, <DegreeProgram.ElectronicsEngg: 'BS Electronics Engineering'>, <DegreeProgram.IndustrialEngg: 'BS Industrial Engineering'>, <DegreeProgram.MechEngg: 'BS Mechanical Engineering'>]
Interviewees: 35
Interviewee Info: 
 Name: David Alfonso Valenzuela
 Course: BS Industrial Engineering
Found previous instance of David Alfonso Valenzuela. Using that instead
Interviewee Info: 
 Name: Marianne Veronica Punzalan
 Course: BS Computer Engineering
Found previous instance of Marianne Veronica Punzalan. Using that instead
Booked Marianne Veronica Punzalan to Times.Time0900_0945

Interviewee Info: 
 Name: Denzell Robyn Dy
 Course: BS Computer Science
Found previous instance of Denzell Robyn Dy. Using that instead
Interviewee Info: 
 Name: Juan Miguel Azul
 Course: BS

## Dataset Creation

In [226]:
def create_datasheet(day : str, type : str):
    fin_dict = [
    ]
    for interviewer in interviewer_availability_dict[day][type]:
        for timeslot in interviewer.time_slots:
            try: 
                for idx, interviewee in enumerate(interviewer.interviewee_list[timeslot]):
                    fin_dict.append([interviewer.name, timeslot.value, f"{interviewee.name} : {interviewee.capes_id}",])
            except KeyError:
                fin_dict.append([interviewer.name, timeslot.value, 'None'])
    # rc_nov_13_df = pd.DataFrame(
    #     data = fin_dict
    # )
    return pd.DataFrame(columns=['Company', 'Time Slot', 'Interviewee'], data=fin_dict).set_index('Company')


### Create Nov 13 Dataframes

In [227]:

final_df_nov_13_rc = create_datasheet('November 13', 'RC')
final_df_nov_13_is = create_datasheet('November 13', 'IS')
final_df_nov_14_rc = create_datasheet('November 14', 'RC')
final_df_nov_14_is = create_datasheet('November 14', 'IS')

final_df_nov_13_rc.to_csv('nov_13_rc.csv')
final_df_nov_13_is.to_csv('nov_13_is.csv')

final_df_nov_14_rc.to_csv('nov_14_rc.csv')
final_df_nov_14_is.to_csv('nov_14_is.csv')